## Invoice Status

### Imports and Library

In [1]:
import polars as pl
from data_source.make_dataset import load_gsheet_data
from data_source.sheet_ids import INVOICING,INVOICE_STATUS

In [2]:
async def load_invoice_status()->pl.LazyFrame:
    """Invoice status dataset"""
    df = await load_gsheet_data(INVOICING,INVOICE_STATUS)
    return df

In [7]:
def invoice_type_enum()->pl.Enum:
    """Invoice type enum"""
    list_of_invoice_type = [
        "oss",
        "sto",
        "si",
        "monthly"
    ]

    return pl.Enum(list_of_invoice_type)

In [14]:
invoice = await load_invoice_status()
open_invoice = invoice.select(
    pl.col("Month").alias("month"),
    pl.col('Report Type').cast(invoice_type_enum()).alias("type"),
    pl.col("Report Name").str.replace(" - OSS","").alias("name"),
    pl.col("ShipOwner").alias("customer"),
    pl.col("Starting Date").str.to_date(format="%d/%m/%Y").alias("start_date"),
    pl.col("Ending Date").str.to_date(format="%d/%m/%Y",strict=False).alias("end_date"),
    pl.col("Status").alias("status")
).filter(pl.col("status").ne(pl.lit("Closed")))

In [36]:
open_invoice.filter(pl.col("type").eq(pl.lit("oss"))).collect()

month,type,name,customer,start_date,end_date,status
str,enum,str,str,date,date,str
"""May '25""","""oss""","""ATERPE ALAI""","""MAERSKLINE""",2025-04-28,2025-05-01,"""Not Ready"""
"""May '25""","""oss""","""ECHEBASTAR""","""MAERSKLINE""",2025-05-01,2025-05-31,"""Not Ready"""
"""May '25""","""oss""","""INPESCA""","""MAERSKLINE""",2025-05-01,2025-05-31,"""Not Ready"""
"""May '25""","""oss""","""ISLAND CATCH""","""MAERSKLINE""",2025-05-01,2025-05-31,"""Not Ready"""
"""May '25""","""oss""","""OCEAN BASKET""","""MAERSKLINE""",2025-05-01,2025-05-31,"""Not Ready"""
…,…,…,…,…,…,…
"""July '25""","""oss""","""ECHEBASTAR""","""MAERSKLINE""",2025-07-01,2025-07-31,"""Not Ready"""
"""July '25""","""oss""","""ALAKRANA""","""MAERSKLINE""",2025-07-16,null,"""Not Ready"""
"""July '25""","""oss""","""NOUR""","""MAERSKLINE""",2025-07-15,2025-07-18,"""Not Ready"""


In [37]:
from dataframe.stuffing import coa
from datetime import date

"ATERPE ALAI"	"MAERSKLINE"	2025-04-28	2025-05-01

In [48]:
client_name = "ATERPE ALAI"
start_date = date(year=2025,month=4,day=28)
end_date = date(year=2025,month=5,day=1)

In [50]:
msk_plugin = await coa()
msk_plugin.collect().filter(pl.col("vessel_client").eq(client_name).and_(
    pl.col("date_plugged").is_between(start_date,end_date)
))

vessel_client,customer,date_plugged,time_plugged,container_number,operation_type,shipping_line,plugged_status,tonnage,set_point,date_out,location,days_on_plug,plugin_price,monitoring_price,electricity_unit_price,total_electricity,total
str,enum,date,time,enum,str,enum,enum,f64,i64,date,str,i64,f64,f64,f64,f64,f64
"""ATERPE ALAI""","""MAERSKLINE""",2025-04-29,22:29:00,"""MSFU0002572""","""Full OSS""","""MAERSKLINE""","""Full""",24.135,-60,2025-05-06,"""LML""",8,25.0,30.0,110.0,880.0,935.0
"""ATERPE ALAI""","""MAERSKLINE""",2025-04-29,20:05:00,"""MSFU0007893""","""Full OSS""","""MAERSKLINE""","""Full""",24.49,-60,2025-05-06,"""LML""",8,25.0,30.0,110.0,880.0,935.0
"""ATERPE ALAI""","""MAERSKLINE""",2025-04-29,20:19:00,"""MSFU0013284""","""Full OSS""","""MAERSKLINE""","""Full""",26.645,-60,2025-05-06,"""LML""",8,25.0,30.0,110.0,880.0,935.0
"""ATERPE ALAI""","""MAERSKLINE""",2025-04-29,20:00:00,"""MSFU0014070""","""Full OSS""","""MAERSKLINE""","""Completed""",22.15,-60,2025-05-06,"""LML""",8,25.0,30.0,110.0,880.0,935.0
"""ATERPE ALAI""","""MAERSKLINE""",2025-04-30,12:23:00,"""MNBU0116463""","""Full OSS""","""MAERSKLINE""","""Completed""",7.62,-25,2025-05-06,"""LML""",7,25.0,30.0,70.0,490.0,545.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ATERPE ALAI""","""MAERSKLINE""",2025-04-30,19:20:00,"""MNBU4587477""","""Full OSS""","""MAERSKLINE""","""Partial""",23.89,-25,2025-05-01,"""For Completion""",1,25.0,0.0,70.0,70.0,95.0
"""ATERPE ALAI""","""MAERSKLINE""",2025-04-30,12:21:00,"""MNBU4490445""","""Full OSS""","""MAERSKLINE""","""Completed""",10.48,-25,2025-05-06,"""LML""",7,25.0,30.0,70.0,490.0,545.0
"""ATERPE ALAI""","""MAERSKLINE""",2025-05-01,10:25:00,"""MNBU3133891""","""Full OSS""","""MAERSKLINE""","""Completed""",5.36,-25,2025-05-06,"""LML""",6,25.0,30.0,70.0,420.0,475.0


In [22]:
from dataframe.transport import transfer

In [24]:
msk_transfer = await transfer()
msk_transfer.filter(pl.col("remarks").eq(pl.lit("MAERSKLINE")).and_(
    pl.col("status").eq(pl.lit("Full"))
)).collect()

/tmp/ipykernel_362378/4002768550.py:4: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  )).collect()


day_name,date,container_number,line,movement_type,driver,origin,time_out,destination,time_in,status,type,size,remarks,shifting_price,haulage_price
enum,date,enum,enum,enum,enum,enum,time,enum,time,enum,enum,enum,str,f64,f64
"""Tue""",2024-01-09,"""MNBU0404650""","""MAERSK""","""Delivery""","""IPHS""","""IPHS""",10:00:00,"""LML""",10:10:00,"""Full""","""Reefer""","""40'""","""MAERSKLINE""",0.0,90.0
"""Tue""",2024-01-09,"""MSFU8626370""","""MAERSK""","""Delivery""","""IPHS""","""IPHS""",10:20:00,"""LML""",10:30:00,"""Full""","""Reefer""","""40'""","""MAERSKLINE""",0.0,90.0
"""Tue""",2024-01-09,"""MSFU0002546""","""MAERSK""","""Delivery""","""IPHS""","""IPHS""",08:55:00,"""LML""",09:10:00,"""Full""","""Reefer""","""40'""","""MAERSKLINE""",0.0,90.0
"""Tue""",2024-01-09,"""MNBU9070194""","""MAERSK""","""Delivery""","""IPHS""","""IPHS""",09:30:00,"""LML""",09:40:00,"""Full""","""Reefer""","""40'""","""MAERSKLINE""",0.0,90.0
"""Tue""",2024-01-09,"""MNBU4156119""","""MAERSK""","""Delivery""","""IPHS""","""IPHS""",13:50:00,"""LML""",14:00:00,"""Full""","""Reefer""","""40'""","""MAERSKLINE""",0.0,90.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Tue""",2025-07-22,"""SUDU6037140""","""MAERSK""","""Delivery""","""IPHS""","""IPHS""",12:45:00,"""LML""",12:55:00,"""Full""","""Reefer""","""40'""","""MAERSKLINE""",0.0,90.0
"""Tue""",2025-07-22,"""MWCU5278860""","""MAERSK""","""Delivery""","""IPHS""","""IPHS""",13:30:00,"""LML""",13:40:00,"""Full""","""Reefer""","""40'""","""MAERSKLINE""",0.0,90.0
"""Tue""",2025-07-22,"""SUDU8031666""","""MAERSK""","""Delivery""","""IPHS""","""IPHS""",14:00:00,"""LML""",14:10:00,"""Full""","""Reefer""","""40'""","""MAERSKLINE""",0.0,90.0
